# پروژه Text-to-SQL با بازاندیشی نحوی و معنایی

این نوت‌بوک مراحل ساخت دیتابیس، اجرای دمو، ارزیابی ۹ پرسش و اجرای تست‌ها را ثبت می‌کند. برای اجرای مجدد، نوت‌بوک را از داخل پوشه پروژه باز کنید.

**توجه:** دمو و تست‌های محلی به API Key نیاز ندارند؛ سلول اجرای مدل واقعی در انتها نیازمند `OPENAI_API_KEY` است.

## ۱. ساخت دیتابیس مطابق صورت‌مسئله


In [1]:
!python setup_db.py


Database created successfully: /mnt/data/TextToSQL_Project/bank_database.db
  Customer: 8 rows
  Account: 12 rows
  Transaction: 35 rows
  Transfer: 7 rows


## ۲. اجرای دو سناریوی بازاندیشی
سناریوی اول خطای نام جدول و سناریوی دوم خطای منطقی JOIN را نمایش می‌دهد.


In [2]:
!python app.py --demo



DEMO 1 — SYNTACTIC REFLECTION
Question: Show customers with accounts whose balance is above 10,000.
SQL execution attempts: 2

Final SQL:
SELECT c.FirstName, c.LastName, a.AccountNumber, a.Balance FROM Customer AS c JOIN Account AS a ON c.CustomerID = a.CustomerID WHERE a.Balance > 10000 ORDER BY a.Balance DESC;

Rows returned: 1
[
  {
    "FirstName": "Fatemeh",
    "LastName": "Esfahani",
    "AccountNumber": "ACC600001",
    "Balance": 12500.0
  }
]

Semantic review:
{
  "is_correct": true,
  "feedback": "The repaired query uses the Customer-Account foreign key and the requested balance filter.",
  "corrected_sql": null,
  "review_parse_error": false
}

Execution trace:
  - Generated the initial SQL query.
  - Initial SQL: SELECT FirstName, LastName, Balance FROM CustomerAccount WHERE Balance > 10000 ORDER BY Balance DESC;
  - Syntactic attempt 1: OperationalError: no such table: CustomerAccount
  - Syntactic Reflection requested repair 1.
  - SQL after syntactic repair: SELECT c.F

## ۳. اجرای ۹ پرسش ارزیابی و نمایش خروجی واقعی


In [3]:
!python evaluate_queries.py



Q1 [Easy] — چه داده‌هایی در پایگاه داده وجود دارد؟
SQL returned 4 row(s).
[
  {
    "TableName": "Account"
  },
  {
    "TableName": "Customer"
  },
  {
    "TableName": "Transaction"
  },
  {
    "TableName": "Transfer"
  }
]

Q2 [Easy] — نام و نام خانوادگی مشتریانی که بعد از 01-01-2021 ثبت‌نام کرده‌اند را نشان بده.
SQL returned 4 row(s).
[
  {
    "CustomerID": 6,
    "FirstName": "Fatemeh",
    "LastName": "Esfahani",
    "CreatedAt": "2021-02-14 02:14:14"
  },
  {
    "CustomerID": 1,
    "FirstName": "Ali",
    "LastName": "Rezaei",
    "CreatedAt": "2022-07-20 12:44:28"
  },
  {
    "CustomerID": 8,
    "FirstName": "Maryam",
    "LastName": "Hosseini",
    "CreatedAt": "2022-10-01 07:40:11"
  },
  {
    "CustomerID": 2,
    "FirstName": "Sara",
    "LastName": "Ahmadi",
    "CreatedAt": "2022-12-01 04:14:48"
  }
]

Q3 [Easy] — تمام حساب‌های فعال (Active) را همراه با موجودی نشان بده.
SQL returned 10 row(s).
[
  {
    "AccountID": 7,
    "AccountNumber": "ACC600001",
    "Account

## ۴. تست‌های خودکار


In [4]:
!python -m unittest discover -s tests -v


test_all_nine_reference_queries_execute (test_agent.TextToSQLTests.test_all_nine_reference_queries_execute) ... ok
test_both_sender_and_receiver_customers_are_found (test_agent.TextToSQLTests.test_both_sender_and_receiver_customers_are_found) ... ok
test_database_schema_and_seed_counts_match_assignment (test_agent.TextToSQLTests.test_database_schema_and_seed_counts_match_assignment) ... ok
test_largest_single_transaction (test_agent.TextToSQLTests.test_largest_single_transaction) ... ok
test_multiple_statements_are_blocked (test_agent.TextToSQLTests.test_multiple_statements_are_blocked) ... ok
test_only_withdrawal_account (test_agent.TextToSQLTests.test_only_withdrawal_account) ... ok
test_read_only_select_returns_rows (test_agent.TextToSQLTests.test_read_only_select_returns_rows) ... ok
test_semantic_reflection_corrects_join_once (test_agent.TextToSQLTests.test_semantic_reflection_corrects_join_once) ... ok
test_syntactic_reflection_repairs_missing_table (test_agent.TextToSQLTests.tes

## ۵. اجرای مدل زبانی واقعی
ابتدا `.env.example` را به `.env` کپی و `OPENAI_API_KEY` را تنظیم کنید. سپس سلول زیر را اجرا کنید.


In [ ]:
from dotenv import load_dotenv
load_dotenv()
from pathlib import Path
from agent import TextToSQLAgent
from backend import LangChainBackend
db_path = Path('bank_database.db')
agent = TextToSQLAgent(db_path, LangChainBackend(model_name='gpt-4o-mini'))
result = agent.ask('List active accounts with their balances.')
print(result.final_sql)
print(result.rows)
print('\n'.join(result.trace))


## ۶. تأیید اتصال زنده به OpenAI API
پس از ساخت `.env` با یک کلید تازه و معتبر، ابتدا درخواست کم‌هزینه را اجرا کنید. برای بررسی کامل مسیر Text-to-SQL، آرگومان `--smoke-only` را حذف کنید. کلید را داخل نوت‌بوک ننویسید.


In [ ]:
!python verify_live_api.py --smoke-only
